# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `production_2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv

In [2]:
import dask.dataframe as dd

c:\Users\Alexander\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:17: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(
C:\Users\Alexander\AppData\Local\Temp\ipykernel_23060\676544213.py:1: DeprecationWarning: The current Dask DataFrame implementation is deprecated. 
In a future release, Dask DataFrame will use new implementation that
contains several improvements including a logical query planning.
The user-facing DataFrame API will remain unchanged.

The new implementation is already available and can be enabled by
installing the dask-expr library:

    $ pip install dask-expr

and turning the query planning option on:

    >>> import dask
    >>> dask.config.set({'dataframe.query-planning': True})
    >>> import dask.dataframe as dd

API documentation for the new implementation is available at
https://docs.dask.org/en/stable/dask-expr-api.html

Any feedbac

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
# import packages
import os
from glob import glob

In [4]:
# get price data
path = '../../05_src/data/prices'
PRICE_DATA = os.getenv("PRICE_DATA")

In [5]:
# read dataframe
parquet_files = glob(os.path.join(path, "*/*/*.parquet"))

In [6]:
# convert to data frame
dd_px = dd.read_parquet(parquet_files)
print(len(parquet_files), len(dd_px))

11207 2777702


In [6]:
dd_px.head()

,Date,Open,High,Low,Close,Adj Close,Volume,sector,subsector,year
ticker,,,,,,,,,,
A,2000-01-03,56.330471,56.464592,48.193848,51.502148,43.463043,4674353,Health Care,Life Sciences Tools & Services,2000
A,2000-01-04,48.730328,49.266811,46.316166,47.567955,40.142933,4765083,Health Care,Life Sciences Tools & Services,2000
A,2000-01-05,47.389126,47.567955,43.141991,44.617310,37.652866,5758642,Health Care,Life Sciences Tools & Services,2000
A,2000-01-06,44.080830,44.349072,41.577251,42.918453,36.219185,2534434,Health Care,Life Sciences Tools & Services,2000
A,2000-01-07,42.247852,47.165592,42.203148,46.494991,39.237457,2819626,Health Care,Life Sciences Tools & Services,2000


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Adjusted Close:
    
    - `returns`: (Adj Close / Adj Close_lag) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [7]:
# import numpy
import numpy as np

In [8]:
# add lag variables for close and adj close
dd = dd_px.groupby('ticker').apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1),
                       AdjClose_lag_1 = x['Adj Close'].shift(1))
           )
dd.head()

C:\Users\Alexander\AppData\Local\Temp\ipykernel_23060\1425153208.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  df = dd_px.groupby('ticker').apply(


Date       Open       High        Low      Close  \
ticker ticker                                                          
HUM    HUM    2006-01-03  54.910000  55.779999  54.099998  55.779999   
       HUM    2006-01-04  56.200001  57.639999  56.180000  56.869999   
       HUM    2006-01-05  57.000000  57.880001  56.930000  57.459999   
       HUM    2006-01-06  57.400002  57.500000  56.570000  57.080002   
       HUM    2006-01-09  57.090000  57.450001  56.360001  57.330002   

               Adj Close   Volume       sector            subsector  year  \
ticker ticker                                                               
HUM    HUM     49.805721  2689400  Health Care  Managed Health Care  2006   
       HUM     50.778961  2334200  Health Care  Managed Health Care  2006   
       HUM     51.305786  1911200  Health Care  Managed Health Care  2006   
       HUM     50.966484  1991700  Health Care  Managed Health Care  2006   
       HUM     51.189713  1874500  Health Care  Managed Health Care  2006   

               Close_lag_1  AdjClose_lag_1  
ticker ticker                               
HUM    HUM             NaN             NaN  
       HUM       55.779999       49.805721  
       HUM       56.869999       50.778961  
       HUM       57.459999       51.305786  
       HUM       57.080002       50.966484

In [11]:
# calculate returns
dd_feat = dd.assign(
    returns = lambda x: (x['Adj Close'] / x['AdjClose_lag_1']) - 1,
    hi_lo_range = lambda x: (x['High'] - x['Low'])
)

dd_feat.head()

Date       Open       High        Low      Close  \
ticker ticker                                                          
HUM    HUM    2006-01-03  54.910000  55.779999  54.099998  55.779999   
       HUM    2006-01-04  56.200001  57.639999  56.180000  56.869999   
       HUM    2006-01-05  57.000000  57.880001  56.930000  57.459999   
       HUM    2006-01-06  57.400002  57.500000  56.570000  57.080002   
       HUM    2006-01-09  57.090000  57.450001  56.360001  57.330002   

               Adj Close   Volume       sector            subsector  year  \
ticker ticker                                                               
HUM    HUM     49.805721  2689400  Health Care  Managed Health Care  2006   
       HUM     50.778961  2334200  Health Care  Managed Health Care  2006   
       HUM     51.305786  1911200  Health Care  Managed Health Care  2006   
       HUM     50.966484  1991700  Health Care  Managed Health Care  2006   
       HUM     51.189713  1874500  Health Care  Managed Health Care  2006   

               Close_lag_1  AdjClose_lag_1   returns  hi_lo_range  
ticker ticker                                                      
HUM    HUM             NaN             NaN       NaN     1.680000  
       HUM       55.779999       49.805721  0.019541     1.459999  
       HUM       56.869999       50.778961  0.010375     0.950001  
       HUM       57.459999       51.305786 -0.006613     0.930000  
       HUM       57.080002       50.966484  0.004380     1.090000

+ Convert the Dask data frame to a pandas data frame. 
+ Add a rolling average return calculation with a window of 10 days.
+ *Tip*: Consider using `.rolling(10).mean()`.

(3 pt)

In [12]:
# convert to pandas dataframe
import pandas as pd
df_feat = pd.DataFrame(dd_feat.compute())

In [13]:
# calculate returns rolling average over 10 days
df_feat['avg_returns'] = df_feat['returns'].rolling(10).mean()

df_feat.head(30)

Date       Open       High        Low      Close  \
ticker ticker                                                          
HUM    HUM    2006-01-03  54.910000  55.779999  54.099998  55.779999   
       HUM    2006-01-04  56.200001  57.639999  56.180000  56.869999   
       HUM    2006-01-05  57.000000  57.880001  56.930000  57.459999   
       HUM    2006-01-06  57.400002  57.500000  56.570000  57.080002   
       HUM    2006-01-09  57.090000  57.450001  56.360001  57.330002   
       HUM    2006-01-10  57.340000  57.750000  56.500000  56.709999   
       HUM    2006-01-11  56.799999  58.070000  56.759998  56.770000   
       HUM    2006-01-12  56.430000  57.240002  56.009998  56.849998   
       HUM    2006-01-13  56.480000  56.840000  55.709999  56.529999   
       HUM    2006-01-17  56.049999  56.310001  55.540001  55.660000   
       HUM    2006-01-18  55.660000  57.830002  55.660000  57.669998   
       HUM    2006-01-19  57.919998  58.259998  55.849998  56.939999   
       HUM    2006-01-20  56.950001  56.959999  54.880001  55.180000   
       HUM    2006-01-23  55.270000  55.480000  53.310001  53.950001   
       HUM    2006-01-24  54.000000  55.529999  53.980000  55.169998   
       HUM    2006-01-25  55.549999  56.160000  55.389999  55.669998   
       HUM    2006-01-26  54.700001  55.279999  53.450001  54.900002   
       HUM    2006-01-27  55.299999  56.330002  55.099998  56.049999   
       HUM    2006-01-30  55.900002  56.840000  55.900002  56.400002   
       HUM    2006-01-31  56.049999  56.380001  55.250000  55.770000   
       HUM    2006-02-01  55.889999  57.230000  55.889999  56.560001   
       HUM    2006-02-02  56.500000  57.259998  56.470001  57.000000   
       HUM    2006-02-03  54.389999  55.419998  52.750000  54.900002   
       HUM    2006-02-06  54.910000  54.910000  52.740002  53.790001   
       HUM    2006-02-07  53.950001  54.250000  52.230000  52.840000   
       HUM    2006-02-08  52.000000  52.520000  50.000000  51.330002   
       HUM    2006-02-09  51.580002  52.669998  51.450001  51.480000   
       HUM    2006-02-10  51.549999  52.009998  50.820000  50.849998   
       HUM    2006-02-13  50.060001  50.799999  49.450001  50.419998   
       HUM    2006-02-14  50.480000  52.000000  50.480000  51.660000   

               Adj Close   Volume       sector            subsector  year  \
ticker ticker                                                               
HUM    HUM     49.805721  2689400  Health Care  Managed Health Care  2006   
       HUM     50.778961  2334200  Health Care  Managed Health Care  2006   
       HUM     51.305786  1911200  Health Care  Managed Health Care  2006   
       HUM     50.966484  1991700  Health Care  Managed Health Care  2006   
       HUM     51.189713  1874500  Health Care  Managed Health Care  2006   
       HUM     50.636089  1433000  Health Care  Managed Health Care  2006   
       HUM     50.689693  1488800  Health Care  Managed Health Care  2006   
       HUM     50.761120   924400  Health Care  Managed Health Care  2006   
       HUM     50.475395  1754700  Health Care  Managed Health Care  2006   
       HUM     49.698551  1487100  Health Care  Managed Health Care  2006   
       HUM     51.493286  1492300  Health Care  Managed Health Care  2006   
       HUM     50.841457  2487200  Health Care  Managed Health Care  2006   
       HUM     49.269974  1865200  Health Care  Managed Health Care  2006   
       HUM     48.171715  1960000  Health Care  Managed Health Care  2006   
       HUM     49.261044  1453700  Health Care  Managed Health Care  2006   
       HUM     49.707500  2495600  Health Care  Managed Health Care  2006   
       HUM     49.019978  3227800  Health Care  Managed Health Care  2006   
       HUM     50.046799  1715300  Health Care  Managed Health Care  2006   
       HUM     50.359314  1406400  Health Care  Managed Health Care  2006   
       HUM     49.796787  1791700  Health Care  Managed Health Care  2006   
       HUM     50.502

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was not necessary to convert to a pandas dataframe to calculate the moving average. As mentioned in class, dask is designed to handle lots of data better than pandas. Dask still operates the same as pandas, but is designed to lazy execute. It would have been better to use dask because the calculation would have used less memory and been more efficient. Dask computes the data in chunks, and does not output all the data to see, making it easier for the computer and program to handle. 

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.